In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from sklearn.metrics.pairwise import cosine_similarity


# 1. LOAD TOKENIZER — converts text to integer IDs (no semantics)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")


In [ ]:
faq_texts = [
    {
        "id": "c01",
        "question": "What are the essential elements of a valid contract?",
        "answer": "A valid contract generally requires offer, acceptance, consideration, legal capacity, and a lawful purpose. Both parties must mutually assent to the terms.",
        "category": "Contracts",
    },
    {
        "id": "c02",
        "question": "Is a verbal contract enforceable in court?",
        "answer": "Verbal contracts can be enforceable unless the agreement falls under the Statute of Frauds, which requires certain contracts to be in writing.",
        "category": "Contracts",
    },
    {
        "id": "c03",
        "question": "What happens when one party breaches a contract?",
        "answer": "The non-breaching party may seek remedies such as damages, specific performance, or rescission depending on the contract and jurisdiction.",
        "category": "Contracts",
    },
    {
        "id": "c04",
        "question": "What is indemnification in a service agreement?",
        "answer": "Indemnification is a contractual obligation to compensate another party for losses or damages arising from specified events or claims.",
        "category": "Contracts",
    },
    {
        "id": "c05",
        "question": "Should an employment contract include a non-compete clause?",
        "answer": "Non-compete clauses are common but enforceability varies by jurisdiction. They must be reasonable in scope, duration, and geography to be upheld.",
        "category": "Contracts",
    },
    {
        "id": "c06",
        "question": "What is a limitation of liability clause?",
        "answer": "It caps the amount one party must pay for damages arising from the contract, often excluding liabilities for gross negligence, willful misconduct, or IP infringement.",
        "category": "Contracts",
    },
    {
        "id": "n01",
        "question": "What is an NDA and when should it be used?",
        "answer": "A Non-Disclosure Agreement protects confidential information. Use it before sharing trade secrets, business plans, or proprietary data with third parties.",
        "category": "NDAs",
    },
    {
        "id": "n02",
        "question": "Can an NDA be enforced if the recipient discloses information?",
        "answer": "Yes, if the NDA is well-drafted and reasonable, the disclosing party can seek injunctive relief and monetary damages for unauthorized disclosure.",
        "category": "NDAs",
    },
    {
        "id": "n03",
        "question": "How long does an NDA typically last?",
        "answer": "NDAs often last for 3 to 5 years, but some obligations, such as those covering trade secrets, can survive indefinitely or until the information becomes public.",
        "category": "NDAs",
    },
    {
        "id": "n04",
        "question": "What are key terms to include in a data processing agreement?",
        "answer": "Include roles of parties, data categories, processing purposes, security measures, subprocessor governance, breach notification, and audit rights.",
        "category": "NDAs",
    },
    {
        "id": "ip01",
        "question": "What is the difference between a patent and a trade secret?",
        "answer": "A patent protects an invention publicly for a limited time in exchange for disclosure, while a trade secret remains confidential for as long as it provides competitive value.",
        "category": "Intellectual Property",
    },
    {
        "id": "ip02",
        "question": "How do I register a copyright for my software?",
        "answer": "Copyright arises automatically upon creation, but you can register it with the U.S. Copyright Office to obtain public record and the ability to sue for statutory damages.",
        "category": "Intellectual Property",
    },
    {
        "id": "ip03",
        "question": "What constitutes trademark infringement?",
        "answer": "Trademark infringement occurs when an unauthorized party uses a mark that is likely to confuse consumers about the source of goods or services.",
        "category": "Intellectual Property",
    },
    {
        "id": "ip04",
        "question": "Can I use open-source code in a commercial product?",
        "answer": "It depends on the license. Permissive licenses like MIT and Apache 2.0 usually allow commercial use, while GPL-style licenses may require sharing derivative source code.",
        "category": "Intellectual Property",
    },
    {
        "id": "ip05",
        "question": "How can a company protect its branding internationally?",
        "answer": "Companies can file trademarks in target jurisdictions or use the Madrid Protocol to seek protection in multiple countries through a single application.",
        "category": "Intellectual Property",
    },
]


In [ ]:
# 2. TOKENIZE — chop text into IDs, pad/truncate to 512 tokens
encoded = tokenizer(
    faq_texts,
    padding="max_length",
    truncation=True,
    max_length=512,
    return_tensors="pt",
)
# Shape: [num_samples, 512] — every FAQ padded or truncated to equal length



In [ ]:
# 3. LOAD EMBEDDING MODEL — 4-bit quantized for memory efficiency

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
model = AutoModel.from_pretrained(
    "meta-llama/Llama-3.2-3B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",  # ~2.5 GB VRAM instead of ~16 GB
)


In [ ]:
# 4. MEAN POOLING — average token vectors into one document vector
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked = last_hidden_state * mask
    summed = masked.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts  # Shape: [batch_size, 1536]


In [ ]:
# 5. GENERATE EMBEDDINGS — batch processing to avoid OOM
with torch.no_grad():
    for i in range(0, num_samples, 4):
        outputs = model(input_ids=batch_ids, attention_mask=batch_mask)
        batch_emb = mean_pool(outputs.last_hidden_state, batch_mask)
        all_embeddings.append(batch_emb.cpu())

embeddings = torch.cat(all_embeddings, dim=0).numpy()  # Shape: [15, 1536]


In [ ]:
# 6. VALIDATE — cosine similarity within categories

for category in ["Contracts", "NDAs", "IP"]:
    idx = df[df["category"] == category].index
    sim = cosine_similarity(embeddings[idx])
    mean_sim = (sim.sum() - len(idx)) / (len(idx) * (len(idx) - 1))
    print(f"{category}: within-group similarity = {mean_sim:.4f}")